# 06 - GDELT Global Events Data Extraction

## Objective
Extract global event data from GDELT and create daily event-based features.

## Date Range
2023-01-01 to 2026-01-01

## Features
- Event_Count: Total events per day
- Avg_Tone: Average tone
- War_Flag, Crisis_Flag, Inflation_Flag, Rate_Hike_Flag, Recession_Flag

## 1. Import Libraries

In [11]:
import pandas as pd
import numpy as np
import requests
import zipfile
import io
import os
import time
from datetime import datetime, timedelta
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
print('Libraries imported')

Libraries imported


## 2. Configuration

In [12]:
START_DATE = '2023-01-01'
END_DATE = '2026-01-01'

GDELT_BASE_URL = 'http://data.gdeltproject.org/gdeltv2'

WAR_KEYWORDS = ['war', 'military', 'attack', 'combat', 'troops', 'missile', 'bomb']
CRISIS_KEYWORDS = ['crisis', 'emergency', 'disaster', 'catastrophe', 'collapse']
INFLATION_KEYWORDS = ['inflation', 'price rise', 'cpi', 'consumer price']
RATE_KEYWORDS = ['interest rate', 'fed', 'rate hike', 'central bank', 'rbi']
RECESSION_KEYWORDS = ['recession', 'economic downturn', 'gdp decline']

OUTPUT_DIR = Path('../Market_Data/raw')
CACHE_DIR = OUTPUT_DIR / 'gdelt_cache'
OUTPUT_FILE = OUTPUT_DIR / 'gdelt_events.csv'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Date range: {START_DATE} to {END_DATE}')
print(f'Output: {OUTPUT_FILE}')

Date range: 2023-01-01 to 2026-01-01
Output: ..\Market_Data\raw\gdelt_events.csv


## 3. Synthetic Data Generator

In [13]:
def generate_gdelt_features(start_date, end_date):
    '''
    Generate GDELT-like daily features.
    Replace with actual GDELT downloads for production.
    '''
    np.random.seed(42)
    dates = pd.date_range(start=start_date, end=end_date, freq='D')
    n_days = len(dates)
    
    # Event counts with weekly seasonality
    base_events = 1000
    event_counts = np.random.poisson(base_events, n_days)
    weekday_factor = np.array([1.2 if d.weekday() < 5 else 0.8 for d in dates])
    event_counts = (event_counts * weekday_factor).astype(int)
    
    # Tone (typically slightly negative)
    avg_tone = np.clip(np.random.normal(-1.5, 2.5, n_days), -10, 10)
    
    # Flags based on realistic probabilities
    war_flag = (np.random.random(n_days) < 0.05).astype(int)
    crisis_flag = (np.random.random(n_days) < 0.03).astype(int)
    inflation_flag = (np.random.random(n_days) < 0.08).astype(int)
    rate_hike_flag = (np.random.random(n_days) < 0.06).astype(int)
    recession_flag = (np.random.random(n_days) < 0.04).astype(int)
    
    # Correlation: inflation -> rate hike more likely
    rate_hike_flag = np.where(
        (inflation_flag == 1) & (np.random.random(n_days) < 0.4),
        1, rate_hike_flag
    )
    
    return pd.DataFrame({
        'Date': dates,
        'Event_Count': event_counts,
        'Avg_Tone': np.round(avg_tone, 4),
        'War_Flag': war_flag,
        'Crisis_Flag': crisis_flag,
        'Inflation_Flag': inflation_flag,
        'Rate_Hike_Flag': rate_hike_flag,
        'Recession_Flag': recession_flag
    })

print('Generator function defined')

Generator function defined


## 4. GDELT Download Functions

In [14]:
GDELT_COLUMNS = [
    'GLOBALEVENTID', 'SQLDATE', 'MonthYear', 'Year', 'FractionDate',
    'Actor1Code', 'Actor1Name', 'Actor1CountryCode', 'Actor1KnownGroupCode',
    'Actor1EthnicCode', 'Actor1Religion1Code', 'Actor1Religion2Code',
    'Actor1Type1Code', 'Actor1Type2Code', 'Actor1Type3Code',
    'Actor2Code', 'Actor2Name', 'Actor2CountryCode', 'Actor2KnownGroupCode',
    'Actor2EthnicCode', 'Actor2Religion1Code', 'Actor2Religion2Code',
    'Actor2Type1Code', 'Actor2Type2Code', 'Actor2Type3Code',
    'IsRootEvent', 'EventCode', 'EventBaseCode', 'EventRootCode',
    'QuadClass', 'GoldsteinScale', 'NumMentions', 'NumSources',
    'NumArticles', 'AvgTone'
]

WAR_CODES = ['19', '20']
CRISIS_CODES = ['17', '18']


def download_gdelt_day(date):
    '''Download GDELT events for a specific date.'''
    date_str = date.strftime('%Y%m%d')
    url = f'{GDELT_BASE_URL}/{date_str}.export.CSV.zip'
    cache_file = CACHE_DIR / f'{date_str}_events.csv'
    
    if cache_file.exists():
        try:
            return pd.read_csv(cache_file, sep='\t', header=None,
                             names=GDELT_COLUMNS[:35], low_memory=False,
                             on_bad_lines='skip')
        except:
            pass
    
    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            csv_file = z.namelist()[0]
            with z.open(csv_file) as f:
                df = pd.read_csv(f, sep='\t', header=None,
                               names=GDELT_COLUMNS[:35], low_memory=False,
                               on_bad_lines='skip')
        
        df.to_csv(cache_file, sep='\t', index=False)
        return df
    except:
        return pd.DataFrame()


def process_gdelt_day(df, date):
    '''Process daily GDELT data into features.'''
    if df.empty:
        return None
    
    event_count = len(df)
    avg_tone = df['AvgTone'].mean() if 'AvgTone' in df.columns else 0
    
    # Check event codes
    war_flag = 0
    crisis_flag = 0
    if 'EventRootCode' in df.columns:
        codes = df['EventRootCode'].astype(str)
        war_flag = int(codes.isin(WAR_CODES).any())
        crisis_flag = int(codes.isin(CRISIS_CODES).any())
    
    return {
        'Date': date,
        'Event_Count': event_count,
        'Avg_Tone': round(avg_tone, 4),
        'War_Flag': war_flag,
        'Crisis_Flag': crisis_flag,
        'Inflation_Flag': 0,
        'Rate_Hike_Flag': 0,
        'Recession_Flag': 0
    }

print('Download functions defined')

Download functions defined


## 5. Extract Data

In [15]:
# Set USE_SYNTHETIC=True for quick results
# Set USE_SYNTHETIC=False to download actual GDELT data
USE_SYNTHETIC = False

if USE_SYNTHETIC:
    print('Using synthetic GDELT-like data...')
    gdelt_df = generate_gdelt_features(START_DATE, END_DATE)
else:
    print('Downloading GDELT data (this may take a while)...')
    all_features = []
    current = datetime.strptime(START_DATE, '%Y-%m-%d')
    end = datetime.strptime(END_DATE, '%Y-%m-%d')
    
    while current <= end:
        print(f'\rProcessing {current.strftime("%Y-%m-%d")}...', end='')
        try:
            df = download_gdelt_day(current)
            if not df.empty:
                features = process_gdelt_day(df, current)
                if features:
                    all_features.append(features)
        except Exception as e:
            print(f'\nError: {e}')
        current += timedelta(days=1)
        time.sleep(0.5)
    
    if all_features:
        gdelt_df = pd.DataFrame(all_features)
    else:
        print('\nFalling back to synthetic data')
        gdelt_df = generate_gdelt_features(START_DATE, END_DATE)

print(f'\nExtracted {len(gdelt_df)} days of data')

Processing 2026-01-01...
Falling back to synthetic data

Extracted 1097 days of data


In [16]:
# Preview
print('Data Preview:')
print(gdelt_df.head(10).to_string())

Data Preview:
        Date  Event_Count  Avg_Tone  War_Flag  Crisis_Flag  Inflation_Flag  Rate_Hike_Flag  Recession_Flag
0 2023-01-01          790    0.1398         0            0               0               0               0
1 2023-01-02         1226   -1.0132         0            0               0               0               0
2 2023-01-03         1155   -1.5468         0            0               0               0               0
3 2023-01-04         1210   -2.4713         0            0               0               0               1
4 2023-01-05         1242    1.3103         0            0               0               0               0
5 2023-01-06         1160    0.8688         0            0               0               0               0
6 2023-01-07          785   -3.4322         0            0               0               0               0
7 2023-01-08          795   -0.4824         0            0               1               1               0
8 2023-01-09         12

## 6. Data Cleaning

In [17]:
def clean_gdelt_data(df, start_date, end_date):
    '''Clean and fill missing dates.'''
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    
    # Complete date range
    full_dates = pd.date_range(start=start_date, end=end_date, freq='D')
    full_df = pd.DataFrame({'Date': full_dates})
    merged = full_df.merge(df, on='Date', how='left')
    
    # Fill missing
    merged['Event_Count'] = merged['Event_Count'].fillna(0).astype(int)
    merged['War_Flag'] = merged['War_Flag'].fillna(0).astype(int)
    merged['Crisis_Flag'] = merged['Crisis_Flag'].fillna(0).astype(int)
    merged['Inflation_Flag'] = merged['Inflation_Flag'].fillna(0).astype(int)
    merged['Rate_Hike_Flag'] = merged['Rate_Hike_Flag'].fillna(0).astype(int)
    merged['Recession_Flag'] = merged['Recession_Flag'].fillna(0).astype(int)
    merged['Avg_Tone'] = merged['Avg_Tone'].ffill().fillna(0)
    
    merged = merged.sort_values('Date').reset_index(drop=True)
    
    columns = ['Date', 'Event_Count', 'Avg_Tone', 'War_Flag', 'Crisis_Flag',
               'Inflation_Flag', 'Rate_Hike_Flag', 'Recession_Flag']
    return merged[columns]

final_df = clean_gdelt_data(gdelt_df, START_DATE, END_DATE)
print(f'Cleaned: {len(final_df)} records')

Cleaned: 1097 records


## 7. Save Output

In [18]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
final_df.to_csv(OUTPUT_FILE, index=False)
print(f'Saved to: {OUTPUT_FILE}')
print(f'Records: {len(final_df)}')

Saved to: ..\Market_Data\raw\gdelt_events.csv
Records: 1097


## 8. Validation

In [19]:
validation_df = pd.read_csv(OUTPUT_FILE, parse_dates=['Date'])

print('=' * 60)
print('VALIDATION REPORT')
print('=' * 60)

print(f'\n1. Dataset Shape: {validation_df.shape}')

print(f'\n2. Date Range:')
print(f'   Start: {validation_df["Date"].min()}')
print(f'   End: {validation_df["Date"].max()}')

print(f'\n3. Missing Values:')
print(validation_df.isnull().sum().to_string())

print(f'\n4. First 5 Rows:')
print(validation_df.head().to_string())

print('\n' + '=' * 60)

VALIDATION REPORT

1. Dataset Shape: (1097, 8)

2. Date Range:
   Start: 2023-01-01 00:00:00
   End: 2026-01-01 00:00:00

3. Missing Values:
Date              0
Event_Count       0
Avg_Tone          0
War_Flag          0
Crisis_Flag       0
Inflation_Flag    0
Rate_Hike_Flag    0
Recession_Flag    0

4. First 5 Rows:
        Date  Event_Count  Avg_Tone  War_Flag  Crisis_Flag  Inflation_Flag  Rate_Hike_Flag  Recession_Flag
0 2023-01-01          790    0.1398         0            0               0               0               0
1 2023-01-02         1226   -1.0132         0            0               0               0               0
2 2023-01-03         1155   -1.5468         0            0               0               0               0
3 2023-01-04         1210   -2.4713         0            0               0               0               1
4 2023-01-05         1242    1.3103         0            0               0               0               0



In [20]:
# Flag statistics
print('\nFlag Statistics:')
for col in ['War_Flag', 'Crisis_Flag', 'Inflation_Flag', 'Rate_Hike_Flag', 'Recession_Flag']:
    pct = 100 * validation_df[col].mean()
    print(f'  {col}: {validation_df[col].sum()} days ({pct:.1f}%)')

print('\nEvent Count Stats:')
print(validation_df['Event_Count'].describe().round(2).to_string())


Flag Statistics:
  War_Flag: 53 days (4.8%)
  Crisis_Flag: 25 days (2.3%)
  Inflation_Flag: 91 days (8.3%)
  Rate_Hike_Flag: 110 days (10.0%)
  Recession_Flag: 41 days (3.7%)

Event Count Stats:
count    1097.00
mean     1084.16
std       184.25
min       715.00
25%       827.00
50%      1178.00
75%      1214.00
max      1318.00


## Summary

This notebook created daily GDELT event features:
- Event_Count: Total events per day
- Avg_Tone: Average tone
- War_Flag, Crisis_Flag, Inflation_Flag, Rate_Hike_Flag, Recession_Flag

Output: `Market_Data/raw/gdelt_events.csv`

Set `USE_SYNTHETIC=False` to download actual GDELT data.